# A/B Testing Analysis - E-commerce Dataset


## Data Loading

Customer behavior, events, and transaction data are combined to analyze an A/B experiment.


In [ ]:
import pandas as pd

customers = pd.read_csv('data/customers.csv')
events = pd.read_csv('data/events.csv')
transactions = pd.read_csv('data/transactions.csv')
products = pd.read_csv('data/products.csv')
campaigns = pd.read_csv('data/campaigns.csv')

## Data Preparation

Conversion flag created (1 = purchase, 0 = no purchase)  
Events merged with transactions  
Missing values filled with 0  

This step prepares a unified dataset to identify which users converted.


In [ ]:
customers.head()
events.head()
transactions.head()

In [ ]:
# purchase yapanlar
transactions['converted'] = 1

In [ ]:
df = events.merge(
    transactions[['customer_id', 'converted']],
    on='customer_id',
    how='left'
)

In [ ]:
df['converted'] = df['converted'].fillna(0)

In [ ]:
df[['customer_id','experiment_group','converted']].head()

## User-Level Dataset

Each user is assigned a single group and conversion flag.

Users may appear multiple times due to event-level data.  
To avoid duplication, we aggregate data at the user level.

In [ ]:
df_user = df.groupby('customer_id').agg({
    'experiment_group': lambda x: x.value_counts().idxmax(),
    'converted': 'max'
}).reset_index()

## Validation

Check data consistency.

Ensure each user appears once and conversion is correctly assigned.

In [ ]:
df_user.head()

## Conversion Rate

Compare groups.

This shows how each experiment group performs in terms of conversion.

In [ ]:
df_user.groupby('experiment_group')['converted'].mean()

In [ ]:
df_user['experiment_group'].value_counts()

## Lift

Measure performance vs Control.

Lift shows how much a variant improves or worsens conversion compared to the baseline.

In [ ]:
cr = df_user.groupby('experiment_group')['converted'].mean()

lift_A = (cr['Variant_A'] - cr['Control']) / cr['Control']
lift_B = (cr['Variant_B'] - cr['Control']) / cr['Control']

print("Lift Variant_A:", lift_A)
print("Lift Variant_B:", lift_B)

In [ ]:
summary = pd.DataFrame({
    'Conversion Rate': cr,
})

summary.loc['Variant_A', 'Lift vs Control'] = lift_A
summary.loc['Variant_B', 'Lift vs Control'] = lift_B

summary

## Key Findings

Control group has the highest conversion rate.

Both Variant_A and Variant_B perform worse than Control.

Differences are small but consistently negative for variants.

## Statistical Test

Check significance.

This test determines whether observed differences are real or due to randomness.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
# başarı sayısı (converted = 1 olanlar)
success = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].sum(),
    df_user[df_user['experiment_group'] == 'Variant_A']['converted'].sum()
]

# toplam user sayısı
nobs = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].count(),
    df_user[df_user['experiment_group'] == 'Variant_A']['converted'].count()
]

stat, p_value = proportions_ztest(success, nobs)

print("Z-stat:", stat)
print("P-value:", p_value)

In [ ]:
success = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].sum(),
    df_user[df_user['experiment_group'] == 'Variant_B']['converted'].sum()
]

nobs = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].count(),
    df_user[df_user['experiment_group'] == 'Variant_B']['converted'].count()
]

stat, p_value = proportions_ztest(success, nobs)

print("Z-stat:", stat)
print("P-value:", p_value)

## Conclusion

Control performs better, but differences are not statistically significant.

No rollout recommended.

Further testing with stronger variations is needed.

## Key Insight

Proper user-level aggregation is critical in A/B testing.

Incorrect grouping can lead to misleading results.